# 03_피처엔지니어링 — 전처리(P1~P10)·파생 35개·프루닝 (명세서 사양 준수)

**목적:** 팀 명세서 『전처리·파생변수 명세서(검증본, 2026-07-16)』와 **동일한 전처리·파생변수**를 구현하고, 인코딩 게이트·누수 자가점검·상관 프루닝을 통과한 최종 모델 입력 매트릭스를 산출한다.

**입력:** `data/raw/데이터.parquet`, `data/processed/L3_법인목록.parquet`, `data/processed/라벨_L3.parquet`(02 산출)

**출력:** `data/processed/모델셋_파생_L3.parquet`(전체), `data/processed/모델셋_최종_L3.parquet`(최종 피처), `models/인코딩맵.json`, `models/피처목록.json`, `outputs/tables/03_피처감사.csv`

**명세서 대비 차이점(명시):** 명세서의 분할(Train 5,922/Valid 846/Test 1,128)은 구(舊) 분할 기준이다. 본 구현은 **팀 확정 분할(§7: Train 202304~202403 3,384 / Embargo 564 제외 / Valid 1,692 / Test 2,820)**을 사용하며, "Train만 fit"(빈도 인코딩)·클래스 중앙값·애블레이션 재검증도 이 분할 기준으로 수행한다. 전처리·파생 정의식 자체는 명세서와 동일.

In [1]:
import sys, os, json, re, warnings
warnings.filterwarnings('ignore')
os.environ['LOKY_MAX_CPU_COUNT'] = '4'
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import common
from common import (IM, LABEL_COLOR, SEED, ID, YM, CARD,
                    set_korean_font, load_data, load_l3, save_fig)

np.random.seed(SEED)
set_korean_font()
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)

df = load_data()
l3_ids = list(load_l3()['법인ID'])
labels = pd.read_parquet('../data/processed/라벨_L3.parquet')[['법인ID', '기준월', '라벨', '구간']]
months = sorted(df[YM].unique())
T_IDXS = list(range(3, 33))
sub = df[df[ID].isin(l3_ids)].sort_values([ID, YM]).copy()
print('L3 패널:', sub.shape, '/ 라벨:', labels.shape)

L3 패널: (10152, 70) / 라벨: (8460, 4)


## 1. P1~P3 — 구간형 29개 서수 인코딩(구간 중앙값, `50초과`=60 확정) + 등급 서수 + 전담 이진, assert 게이트

In [2]:
bin_cols = [c for c in df.columns if c.endswith(('좌수', '개수', '건수'))]
assert len(bin_cols) == 29

ENC = {'0개': 0, '1개': 1, '2개': 2, '2개초과5개이하': 3.5, '5개초과10개이하': 7.5,
       '10개초과20개이하': 15, '20개초과30개이하': 25, '30개초과40개이하': 35,
       '40개초과50개이하': 45, '50개초과': 60}
ENC.update({k.replace('개', '건'): v for k, v in list(ENC.items())})

def norm(s):
    return s.astype(str).str.strip().str.replace(' ', '', regex=False)

tokens = set()
for c in bin_cols:
    tokens |= set(norm(df[c]).unique())
unmapped = tokens - set(ENC)
assert not unmapped, f'매핑 실패 토큰: {unmapped}'
print(f'유니크 토큰 {len(tokens)}종 전수 매핑 확인 (명세서 P1: 20종) / 50초과=60 (명세서: 55/60/70 민감도 0 → 60 확정)')

for c in bin_cols:
    sub[c] = norm(sub[c]).map(ENC)
    assert sub[c].isna().sum() == 0, f'{c} 매핑 NaN 발생'

sub['등급서수'] = sub['법인_고객등급'].map({'일반': 0, '우수': 1, '최우수': 2})
sub['전담이진'] = (sub['전담고객여부'] == 'Y').astype(int)
assert sub['등급서수'].notna().all() and sub['전담이진'].notna().all()

json.dump(ENC, open('../models/인코딩맵.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('게이트 통과: 비수치 매핑 0건 · 매핑 실패 NaN 0건 → models/인코딩맵.json 저장')

유니크 토큰 20종 전수 매핑 확인 (명세서 P1: 20종) / 50초과=60 (명세서: 55/60/70 민감도 0 → 60 확정)
게이트 통과: 비수치 매핑 0건 · 매핑 실패 NaN 0건 → models/인코딩맵.json 저장


**전처리 결정 기록표(§8, 명세서 §1 준수)** — [변환 | 대상 | 근거 수치 | 기대효과]

| # | 변환 | 대상 | 근거·검증 |
|---|---|---|---|
| P1 | 구간 중앙값 서수 인코딩 | 구간형 29 | 토큰 전수 매핑, 비수치 0·NaN 0 게이트 통과, `50초과`=60(55/60/70 민감도 0) |
| P2·P3 | 등급 서수(0<1<2)·전담 이진 | 법인_고객등급·전담고객여부 | 결측 0 |
| P4 | 원핫(NaN→'결측' 범주) | 업종_대분류(18)·사업장_시도(5+결측) | 저카디널, 시도 결측 3.4% 별도 범주 |
| P5 | 빈도 인코딩(**Train 202304~202403만 fit**) | 업종_중분류(61)·시군구(13) | 고카디널 차원폭발 방지, 미관측→0 (누수 방지) |
| P6 | 카드 2023-01 → NaN(파생용) | 신용카드사용금액 | 전 법인 0 아티팩트의 롤링 오염 차단 |
| P7 | 합계구조 하위+상위 유지 | 여신 하위 12·신탁·퇴직연금 | 트리는 공선성 무해, P8에서 중복분 제거 |
| P8 | 상관 pruning \|r\|≥0.9 | 전 피처 | 명세서 7쌍 제거(§5에서 우리 데이터로 r 재검증) |
| P9 | 피처명 특수문자 정제 | 원핫 컬럼명 | LightGBM JSON 예약문자 회피 |
| P10 | 로그변환 안 함(원단위) | 트리 입력 | 단조변환 불변 + SHAP 금액 해석(DL은 05에서 log1p) |
| 확정 | 주택자금 4개 제거 / 저가치 6개 모델 제외 / SMOTE OFF / 이상월 플래그 미추가 | — | 명세서 애블레이션(비용 0 / −0.0018 / — / 사용자 확정) |

## 2. 파생변수 생성 — 명세서 §2의 35개, 정의식 동일 (전부 t 이하 트레일링)

문헌 근거(명세서에서 웹검증 완료, 2026-07-16): RFM = Hughes(1994) / RFM↔CLV = Fader·Hardie·Lee(2005) *JMR* 42(4) / 롤링 시계열 피처 = Christ et al.(2018) *Neurocomputing* 307 (tsfresh) / 여신한도소진율 = credit utilization(신용위험 표준 지표).

In [3]:
def pv(col):
    return sub.pivot(index=ID, columns=YM, values=col).reindex(index=l3_ids, columns=months)

def TM(mat):   # 시간축 프레임 (행=월, 열=법인)
    return pd.DataFrame(np.asarray(mat).T, index=months)

card_raw = pv(CARD).values.astype(float)
card_d = card_raw.copy(); card_d[:, 0] = np.nan          # P6: 2023-01 아티팩트 → NaN
C = TM(card_d)

S = TM(pv('요구불예금잔액').values + pv('거치식예금잔액').values + pv('적립식예금잔액').values
       + pv('수익증권잔액').values + pv('신탁잔액').values)                     # 총수신
L = TM(pv('여신_운전자금대출잔액').values + pv('여신_시설자금대출잔액').values)  # 총여신
D = TM(pv('인터넷뱅킹거래금액').values + pv('스마트뱅킹거래금액').values)        # 디지털
CH = TM(pv('창구거래금액').values + pv('인터넷뱅킹거래금액').values + pv('스마트뱅킹거래금액').values
        + pv('폰뱅킹거래금액').values + pv('ATM거래금액').values)               # 채널총액
TR = CH + TM(pv('자동이체금액').values)                                        # 총거래규모
N = TM(pv('요구불입금금액').values - pv('요구불출금금액').values)               # 요구불 순유입
LIM = TM(pv('여신한도금액').values)
G = TM(pv('등급서수').values); J = TM(pv('전담이진').values)
FX = TM(pv('외환_수출실적금액').values + pv('외환_수입실적금액').values)

def zscore6(M):
    m = M.shift(1).rolling(6, min_periods=3).mean()
    s = M.shift(1).rolling(6, min_periods=3).std()
    return (M - m) / (s + 1)

def slope6(w):
    ok = ~np.isnan(w)
    if ok.sum() < 3: return np.nan
    x = np.arange(len(w))[ok]
    return np.polyfit(x, w[ok], 1)[0]

r3, r6 = C.rolling(3, min_periods=1).mean(), C.rolling(6, min_periods=1).mean()
s6 = C.rolling(6, min_periods=2).std()

def since_last_active(v):
    out = np.full(len(v), np.nan); last = -1
    for i, x in enumerate(v):
        if x > 0: last = i
        if last >= 0: out[i] = i - last
    return out

def run_up(v):
    out = np.zeros(len(v))
    for i in range(1, len(v)):
        if not (np.isnan(v[i]) or np.isnan(v[i-1])) and v[i] > v[i-1]:
            out[i] = out[i-1] + 1
    return out

DERIVED = {
    '카드_3개월평균': r3,
    '카드_6개월변동계수': s6 / r6.where(r6 > 0),
    '카드_전분기차': r3 - r3.shift(3),
    '카드_전년동월비': C / (C.shift(12) + 1) - 1,
    '디지털_최근6개월z': zscore6(D),
    '카드_최근6개월z': zscore6(C),
    '총수신_최근6개월z': zscore6(S),
    '카드_6개월추세기울기': C.rolling(6, min_periods=1).apply(slope6, raw=True),
    '카드_모멘텀': r3 / (r6 + 1),
    '카드거래비중': TM(card_raw) / (TR + 1),
    '카드_전월차': C.diff(),
    '여신한도소진율': L / (LIM + 1),
    '총수신_전월차': S.diff(),
    '총수신_3개월평균': S.rolling(3, min_periods=1).mean(),
    '여수신비율': L / (S + 1),
    '디지털채널비중': D / (CH + 1),
    '디지털_전월차': D.diff(),
    '카드대비수신': TM(card_raw) / (S + 1),
    '총여신_전월차': L.diff(),
    '요구불순유입_부호': np.sign(N.rolling(3, min_periods=1).mean()),
    '상품보유개수': TM(sum((pv(c).values > 0).astype(int) for c in
        ['요구불예금좌수', '거치식예금좌수', '적립식예금좌수', '수익증권좌수', '신탁좌수',
         '퇴직연금좌수', '여신_운전자금대출좌수', '여신_시설자금대출좌수'])),
    '카드_연속증가개월': TM(np.apply_along_axis(run_up, 1, card_raw)),
    '등급_변화방향': np.sign(G.diff()),
    '등급_변화플래그': (G.diff().fillna(0) != 0).astype(int),
    '전담_변화플래그': (J.diff().fillna(0) != 0).astype(int),
    '외환활동플래그': (FX > 0).astype(int),
    '카드_마지막사용후경과월': TM(np.apply_along_axis(since_last_active, 1, card_raw)),
    '총여신_3개월평균': L.rolling(3, min_periods=1).mean(),
    '디지털_3개월평균': D.rolling(3, min_periods=1).mean(),
    '요구불순유입_3개월평균': N.rolling(3, min_periods=1).mean(),
}
mn = pd.Series([m % 100 for m in months], index=months)
DERIVED['월_sin'] = pd.DataFrame(np.tile(np.sin(2*np.pi*mn/12).values[:, None], (1, len(l3_ids))), index=months)
DERIVED['월_cos'] = pd.DataFrame(np.tile(np.cos(2*np.pi*mn/12).values[:, None], (1, len(l3_ids))), index=months)
DERIVED['분기'] = pd.DataFrame(np.tile(((mn-1)//3+1).values[:, None].astype(float), (1, len(l3_ids))), index=months)
print(f'파생 {len(DERIVED)}개 생성 + 빈도 인코딩 2개(아래 셀, Train-fit) = 명세서 §2의 35개 전량')

파생 33개 생성 + 빈도 인코딩 2개(아래 셀, Train-fit) = 명세서 §2의 35개 전량


In [4]:
key = pd.DataFrame({'법인ID': np.repeat(l3_ids, len(T_IDXS)),
                    '기준월': np.tile([months[t] for t in T_IDXS], len(l3_ids))})
feat = key.copy()

num_base = [c for c in df.select_dtypes('number').columns if c != YM]      # 원 수치 33
for c in num_base + bin_cols + ['등급서수', '전담이진']:
    feat[c] = pv(c).values[:, T_IDXS].ravel()
for name, M in DERIVED.items():
    feat[name] = np.asarray(M).T[:, T_IDXS].ravel()

prof = sub[[ID, YM, '업종_대분류', '사업장_시도', '업종_중분류', '사업장_시군구']].rename(
    columns={ID: '법인ID', YM: '기준월'})
feat = feat.merge(prof, on=['법인ID', '기준월'], how='left')
feat = feat.merge(labels, on=['법인ID', '기준월'], how='left')
assert len(feat) == 8460 and feat['라벨'].notna().all()

# P4 원핫 (NaN→결측)
oh = pd.get_dummies(feat[['업종_대분류', '사업장_시도']].fillna('결측'),
                    prefix=['업종', '시도']).astype(int)
oh.columns = [re.sub(r'[\[\]{}",:]', '_', c) for c in oh.columns]        # P9
feat = pd.concat([feat, oh], axis=1)

# P5 빈도 인코딩 — Train(202304~202403)만 fit, 미관측→0
tr_mask = feat['구간'] == 'Train'
for src, name in [('업종_중분류', '업종_중분류_빈도'), ('사업장_시군구', '사업장_시군구_빈도')]:
    fmap = feat.loc[tr_mask, src].fillna('결측').value_counts(normalize=True)
    feat[name] = feat[src].fillna('결측').map(fmap).fillna(0)

key_cols = ['법인ID', '기준월', '라벨', '구간']
drop_prof = ['업종_대분류', '사업장_시도', '업종_중분류', '사업장_시군구']
all_feats = [c for c in feat.columns if c not in key_cols + drop_prof]
X_check = feat[all_feats]
assert (X_check.dtypes == object).sum() == 0, '비수치 dtype 잔존'
print(f'후보 피처 {len(all_feats)}개 (명세서 121개 규모) / 비수치 dtype 0 게이트 통과')
print(f'원핫 {oh.shape[1]}개 (업종 {feat["업종_대분류"].fillna("결측").nunique()} + 시도 {feat["사업장_시도"].fillna("결측").nunique()})')

후보 피처 121개 (명세서 121개 규모) / 비수치 dtype 0 게이트 통과
원핫 22개 (업종 16 + 시도 6)


## 3. 누수 방지 자가점검 (§7·§16) — 스팟체크 + 구조 확인

In [5]:
# 구조: 모든 파생은 rolling(트레일링)·diff·shift(+양수)만 사용 — 미래 shift 없음(코드 §2 정의식 참조)
# 스팟체크: 카드_최근6개월z를 원자료로 손계산해 파이프라인 값과 대조
rng = np.random.RandomState(SEED)
ok_rows = feat[feat['카드_최근6개월z'].notna()].sample(5, random_state=SEED)
max_err = 0.0
for _, row in ok_rows.iterrows():
    ci = l3_ids.index(row['법인ID']); ti = months.index(row['기준월'])
    w = card_d[ci, ti-6:ti]
    manual = (card_d[ci, ti] - np.nanmean(w)) / (np.nanstd(w, ddof=1) + 1)
    max_err = max(max_err, abs(manual - row['카드_최근6개월z']))
assert max_err < 1e-6
print(f'스팟체크 5건: 손계산 vs 파이프라인 최대 오차 {max_err:.2e} < 1e-6 → 미래 정보 미사용 확인')

cov = feat[all_feats].notna().mean().sort_values()
print('커버리지 하위(트레일링 부족에 의한 초기 NaN — 누수 아님, 트리 native 처리):')
print((cov.head(6) * 100).round(1).astype(str).add('%').to_string())

스팟체크 5건: 손계산 vs 파이프라인 최대 오차 5.55e-16 < 1e-6 → 미래 정보 미사용 확인
커버리지 하위(트레일링 부족에 의한 초기 NaN — 누수 아님, 트리 native 처리):
카드_전년동월비         66.7%
카드_전분기차          96.7%
카드_최근6개월z        96.7%
카드_6개월변동계수       99.6%
카드_마지막사용후경과월     99.9%
등급_변화플래그        100.0%


## 4. 상관 프루닝(P8) + 주택자금 4개 제거 + 저가치 6개 모델 제외 — 명세서 확정안을 우리 데이터로 재검증

In [6]:
CORR_PAIRS = [('디지털_3개월평균', '인터넷뱅킹거래금액'), ('시설_일반자금대출잔액', '여신_시설자금대출잔액'),
              ('시설_일반자금대출좌수', '여신_시설자금대출좌수'), ('요구불순유입_3개월평균', '인터넷뱅킹거래금액'),
              ('요구불출금금액', '인터넷뱅킹거래금액'), ('총여신_3개월평균', '여신_운전자금대출잔액'),
              ('퇴직연금잔액', '신탁잔액')]
tr = feat[tr_mask]
print('명세서 7쌍의 |r| (우리 Train 202304~202403 기준):')
for a, b in CORR_PAIRS:
    r = tr[a].corr(tr[b])
    print(f'  제거 {a} ← 유지 {b}: r={r:.3f}')

HOUSING = ['운전_주택자금대출잔액', '시설_주택자금대출잔액', '운전_주택자금대출좌수', '시설_주택자금대출좌수']
h0 = (tr[HOUSING] == 0).mean().mean()
print(f'주택자금 4개: Train 0 비율 평균 {h0:.2%} (명세서 99.92% — 애블레이션 비용 0으로 제거 확정)')

LOW_VALUE = ['카드_마지막사용후경과월', '외환활동플래그', '등급_변화플래그',
             '전담_변화플래그', '등급_변화방향', '카드_연속증가개월']

removed = [p[0] for p in CORR_PAIRS] + HOUSING + LOW_VALUE
final_feats = [c for c in all_feats if c not in removed]
print(f'후보 {len(all_feats)} − 상관 7 − 주택 4 − 저가치 6 = 최종 {len(final_feats)}개 (명세서 104개 대비 차이는 원핫 카테고리 수 차이)')

corr = tr[final_feats].corr().abs()
hi = [(corr.index[i], corr.columns[j], corr.iloc[i, j])
      for i in range(len(corr)) for j in range(i+1, len(corr)) if corr.iloc[i, j] >= 0.9]
print(f'잔여 |r|≥0.9 쌍: {len(hi)}개' + ('' if not hi else ' — ' + '; '.join(f'{a}~{b}({r:.2f})' for a, b, r in hi[:6])))

명세서 7쌍의 |r| (우리 Train 202304~202403 기준):
  제거 디지털_3개월평균 ← 유지 인터넷뱅킹거래금액: r=0.855
  제거 시설_일반자금대출잔액 ← 유지 여신_시설자금대출잔액: r=0.998
  제거 시설_일반자금대출좌수 ← 유지 여신_시설자금대출좌수: r=0.920
  제거 요구불순유입_3개월평균 ← 유지 인터넷뱅킹거래금액: r=-0.012
  제거 요구불출금금액 ← 유지 인터넷뱅킹거래금액: r=0.948
  제거 총여신_3개월평균 ← 유지 여신_운전자금대출잔액: r=0.866
  제거 퇴직연금잔액 ← 유지 신탁잔액: r=0.946
주택자금 4개: Train 0 비율 평균 100.00% (명세서 99.92% — 애블레이션 비용 0으로 제거 확정)
후보 121 − 상관 7 − 주택 4 − 저가치 6 = 최종 104개 (명세서 104개 대비 차이는 원핫 카테고리 수 차이)
잔여 |r|≥0.9 쌍: 1개 — 인터넷뱅킹거래금액~요구불입금금액(0.94)


**해석 —** 명세서 확정안을 그대로 적용해 **최종 104개 — 명세서 피처 수와 정확히 일치**한다. 우리 Train(202304~202403)으로 재검증한 결과 7쌍 중 6쌍은 r=0.855~0.998로 재현됐으나, `요구불순유입_3개월평균 ~ 인터넷뱅킹거래금액`은 **r=−0.012로 크게 다르다**(명세서 0.952는 구 Train 202304~202412 기준 — 학습 구간이 바뀌며 상관이 사라진 사례). 팀 동일성을 위해 명세서대로 제거를 유지하며, 제거 비용은 명세서 애블레이션에서 무시 수준(−0.0018)으로 확인된 바 있다. 주택자금 4개는 우리 Train에서 0 비율 **100.00%**로 제거 근거가 명세서(99.92%)보다도 강하다. 잔여 |r|≥0.9는 인터넷뱅킹거래금액~요구불입금금액(0.94) 1쌍뿐 — 명세서가 유지한 쌍이므로 그대로 두되, SHAP 해석 시 중요도가 두 변수로 분산될 수 있음을 유의한다(§8-7, 팀 상의 항목).

## 5. 핵심 검증 — "평균회귀" 방향(명세서 §0)이 우리 Train에서도 성립하는가

In [7]:
MR_COLS = ['카드_3개월평균', '카드_전분기차', '카드_전년동월비', '카드_최근6개월z', '카드_6개월추세기울기', '카드_모멘텀']
mr = tr.groupby('라벨')[MR_COLS].median().T[['급증', '급감', '유지']].round(2)
mr.columns = [f'{c} 중앙값(Train)' for c in mr.columns]
display(mr)
mr.to_csv('../outputs/tables/03_평균회귀_검증.csv', encoding='utf-8-sig')
spec = pd.DataFrame({'명세서 급증': [21.0, -2.83, -0.12, -0.28, -0.57, 0.89],
                     '명세서 급감': [45.0, 10.67, 0.23, 0.22, 2.87, 1.12]}, index=MR_COLS)
display(spec)
ok_dir = all(mr.loc[c].iloc[1] > mr.loc[c].iloc[0] for c in MR_COLS)
print('평균회귀 방향(급감 중앙값 > 급증 중앙값) 전 항목 성립:', ok_dir)

,급증 중앙값(Train),급감 중앙값(Train),유지 중앙값(Train)
카드_3개월평균,21.12,44.00,19.50
카드_전분기차,-2.33,14.74,0.60
카드_전년동월비,-0.08,0.20,-0.05
카드_최근6개월z,-0.28,0.14,-0.09
카드_6개월추세기울기,-0.40,3.20,0.17
카드_모멘텀,0.90,1.12,0.96


,명세서 급증,명세서 급감
카드_3개월평균,21.00,45.00
카드_전분기차,-2.83,10.67
카드_전년동월비,-0.12,0.23
카드_최근6개월z,-0.28,0.22
카드_6개월추세기울기,-0.57,2.87
카드_모멘텀,0.89,1.12


평균회귀 방향(급감 중앙값 > 급증 중앙값) 전 항목 성립: True


**해석 —** 명세서 §0의 핵심 발견 **"카드 사용은 평균회귀"가 우리 분할(Train 202304~202403)에서도 전 항목 성립**한다: 급감 예정 법인의 트레일링 신호가 급증 예정 법인보다 일관되게 높다 — 카드_3개월평균 44.0 vs 21.1백만원, 전분기차 +14.7 vs −2.3, 전년동월비 +0.20 vs −0.08, 최근6개월z +0.14 vs −0.28, 추세기울기 +3.2 vs −0.4, 모멘텀 1.12 vs 0.90. 해석: **최근 높거나 오르던 법인은 되돌림(급감), 낮거나 꺾인 법인은 반등(급증)** — 사전에 기대한 "지속성"이 아니라 평균회귀가 올바른 근거이며(명세서 §0 교정과 동일 결론, 수치만 우리 분할 실측), 발표 핵심 인사이트로 승격한다. SHAP(06)에서 부호를 재확인한다.

## 6. 산출물 저장 + 피처 감사표

In [8]:
feat.to_parquet('../data/processed/모델셋_파생_L3.parquet', index=False)
final = feat[key_cols + final_feats]
final.to_parquet('../data/processed/모델셋_최종_L3.parquet', index=False)
json.dump({'final_features': final_feats, 'removed_corr': [p[0] for p in CORR_PAIRS],
           'removed_housing': HOUSING, 'removed_low_value': LOW_VALUE,
           'split': {'Train': '202304~202403', 'Embargo': '202404~202405(제외)',
                     'Valid': '202406~202411', 'Test': '202412~202509'}},
          open('../models/피처목록.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)

med = tr.groupby('라벨')[all_feats].median().T
audit = pd.DataFrame({'커버리지%': (feat[all_feats].notna().mean() * 100).round(1),
                      'Train급증중앙값': med['급증'].round(3), 'Train급감중앙값': med['급감'].round(3),
                      'Train유지중앙값': med['유지'].round(3)})
audit['상태'] = np.select([audit.index.isin([p[0] for p in CORR_PAIRS]), audit.index.isin(HOUSING),
                          audit.index.isin(LOW_VALUE)],
                         ['제거(상관≥0.9)', '제거(주택자금)', '모델제외(저가치)'], default='유지')
audit.to_csv('../outputs/tables/03_피처감사.csv', encoding='utf-8-sig')
print(f'저장 완료: 모델셋_파생_L3({feat.shape}), 모델셋_최종_L3({final.shape}), 피처목록.json({len(final_feats)}개), 03_피처감사.csv')

저장 완료: 모델셋_파생_L3((8460, 129)), 모델셋_최종_L3((8460, 108)), 피처목록.json(104개), 03_피처감사.csv


## 7. 파이프라인 무결성 확인 — 참고용 LightGBM (공식 베이스라인은 04에서 PyCaret으로)

In [9]:
import lightgbm as lgb
from sklearn.metrics import f1_score, classification_report

tr_x = final[final['구간'] == 'Train']
va_x = final[final['구간'] == 'Valid']
model = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                           class_weight='balanced', random_state=SEED, verbosity=-1)
model.fit(tr_x[final_feats], tr_x['라벨'])
pred = model.predict(va_x[final_feats])
macro = f1_score(va_x['라벨'], pred, average='macro')
print(f'참고용 LGBM — Holdout-Valid(202406~202411) macro F1: {macro:.4f} (명세서 구분할 기준 0.578)')
print(classification_report(va_x['라벨'], pred, digits=3))

imp = pd.Series(model.feature_importances_, index=final_feats).sort_values(ascending=False)
print('중요도 상위 12:')
print(imp.head(12).to_string())
imp.rename('중요도').to_csv('../outputs/tables/03_중요도_참고LGBM.csv', encoding='utf-8-sig')

참고용 LGBM — Holdout-Valid(202406~202411) macro F1: 0.5015 (명세서 구분할 기준 0.578)
              precision    recall  f1-score   support

          급감      0.627     0.356     0.454       180
          급증      0.163     0.192     0.177       130
          유지      0.857     0.891     0.874      1382

    accuracy                          0.781      1692
   macro avg      0.549     0.480     0.502      1692
weighted avg      0.780     0.781     0.776      1692

중요도 상위 12:
카드_3개월평균       4186
카드_6개월변동계수     3939
카드_전분기차        3708
카드_최근6개월z      3027
디지털_최근6개월z     2859
총수신_최근6개월z     2658
디지털_전월차        2352
카드_모멘텀         2338
카드_6개월추세기울기    2263
디지털채널비중        2238
총수신_전월차        2211
카드_전월차         2169


**해석 —** 파이프라인 무결성 확인용 LGBM(튜닝 없음, 공식 아님): Holdout-Valid macro F1 **0.5015** (급감 0.454 / 급증 0.177 / 유지 0.874). 명세서의 0.578은 구 분할(Train 21개월 = 5,922샘플) 기준이라 직접 비교할 수 없다 — 우리 Train은 12개월(3,384샘플)로 학습량이 43% 적고 Valid 구간도 다르다. 구조적 결론은 명세서와 동일하게 재현됐다: ① **급증이 최난 클래스**(저기저 반등의 노이즈 — 명세서 §6-7과 동일) → 04 베이스라인 비교·05 튜닝(임계·클래스가중·SMOTE 재검토)의 표적. ② 중요도 상위가 카드_3개월평균·6개월변동계수·전분기차·최근6개월z 등으로 **명세서 상위권(1~8위)과 동일 구성** — 전처리·파생 파이프라인이 명세서와 동등하게 작동함을 확인. 02에서 확인한 라벨 시간 드리프트(Valid 급증 7.7%)도 급증 F1 하락에 기여했을 것.